????? ????????: Izvekov Igor (izveigor@gmail.com)

In [ ]:
import pandas as pd
import pickle
import re
import json
import os

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('russian'))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

import tika
from tika import parser
tika.initVM()

In [ ]:
classification = {
    "???????? ??? ????????????/???????? ????????": 0,
    "???????? ??? ????????????/???????? ???????? ?????": 1,
    "???????? ??? ????????????/???????? ???????": 2,
    "???????? ??? ????????????/???????? ??????": 3,
    "???????? ??? ????????????/???????? ?????-???????": 4,
}

In [ ]:
# ????????? ?????? ? ???????

data = []
with open("classes.json") as json_file:
    json_data = json_file.read()
    try:
        classes = json.loads(json_data)
    except Exception as e:
        raise e

    for index, file in enumerate(os.listdir('./docs'), start=1):
        class_ = classification[classes[file]]
        parsed = parser.from_file(os.path.join('docs', file))
        content = parsed['content']
        data.append([index, content, class_])

dataset = pd.DataFrame(data, columns=['index', 'text', 'target'])
X = dataset['text'].astype('string')
y = dataset.target.astype('int32')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42)

In [ ]:
one_hot_encoder = OneHotEncoder()
y_train = one_hot_encoder.fit_transform([[target] for target in y_train]).toarray()
y_test = one_hot_encoder.transform([[target] for target in y_test]).toarray()
print(y_train[:5])

In [ ]:
# ??????? ?????? ?? ???????? ????????
def cleaning(doc):
    text = ""
    for word in doc:
        token = re.sub("[^?-??-?\n ']+", '', str(word)).lower()
        if token:
            token = re.sub("\n", " ", token)
            text += token
    text = ' '.join(text.split())

    cleaned_field = []
    for word in text.split():
        lemma_ = lemmatizer.lemmatize(word)
        if lemma_ not in stop_words:
            cleaned_field.append(lemma_)

    return cleaned_field

X_train = [cleaning(text) for text in X_train]
X_test = [cleaning(text) for text in X_test]

In [ ]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform([" ".join(o) for o in X_train])
X_test = vectorizer.transform([" ".join(o) for o in X_test])

In [ ]:
pkl_filename = "vectorizer.pkl"
with open(pkl_filename, 'wb') as file:
    pickle.dump(vectorizer, file)

In [ ]:
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)
acc_random_forest = round(clf.score(X_train, y_train) * 100, 2)
print(acc_random_forest)

In [ ]:
pkl_filename = "model.pkl"
with open(pkl_filename, 'wb') as file:
    pickle.dump(clf, file)

In [ ]:
print(y_pred[:5])
print(y_pred_proba[:5])